# 论文 7：使用深度卷积神经网络进行 ImageNet 分类
## Alex Krizhevsky, Ilya Sutskever, Geoffrey E. Hinton（2012）

### AlexNet：开启深度学习革命的 CNN

AlexNet 以 15.3% 的 top-5 错误率赢得 ImageNet 2012，远优于第二名的 26.2%。这篇论文重新激发了研究界对深度学习的兴趣。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import correlate2d

np.random.seed(42)

## 卷积层实现

CNN 的核心构建块

In [ ]:
def relu(x):
    return np.maximum(0, x)

def conv2d(input_image, kernel, stride=1, padding=0):
    """2D 卷积运算
    
    input_image: (H, W) 或 (C, H, W)
    kernel：(out_channels, in_channels, kH, kW)"""
    if len(input_image.shape) == 2:
        input_image = input_image[np.newaxis, :, :]
    
    in_channels, H, W = input_image.shape
    out_channels, _, kH, kW = kernel.shape
    
    # 添加填充
    if padding > 0:
        input_padded = np.pad(input_image, 
                             ((0, 0), (padding, padding), (padding, padding)), 
                             mode='constant')
    else:
        input_padded = input_image
    
    # 输出尺寸
    out_H = (H + 2*padding - kH) // stride + 1
    out_W = (W + 2*padding - kW) // stride + 1
    
    output = np.zeros((out_channels, out_H, out_W))
    
    # 执行卷积
    for oc in range(out_channels):
        for i in range(out_H):
            for j in range(out_W):
                h_start = i * stride
                w_start = j * stride
                
                # 提取图像块
                patch = input_padded[:, h_start:h_start+kH, w_start:w_start+kW]
                
                # 与核卷积
                output[oc, i, j] = np.sum(patch * kernel[oc])
    
    return output

def max_pool2d(input_image, pool_size=2, stride=2):
    '最大池化操作'
    C, H, W = input_image.shape
    
    out_H = (H - pool_size) // stride + 1
    out_W = (W - pool_size) // stride + 1
    
    output = np.zeros((C, out_H, out_W))
    
    for c in range(C):
        for i in range(out_H):
            for j in range(out_W):
                h_start = i * stride
                w_start = j * stride
                
                pool_region = input_image[c, h_start:h_start+pool_size, 
                                         w_start:w_start+pool_size]
                output[c, i, j] = np.max(pool_region)
    
    return output

# 测试卷积
test_image = np.random.randn(1, 8, 8)
test_kernel = np.random.randn(3, 1, 3, 3) * 0.1

conv_output = conv2d(test_image, test_kernel, stride=1, padding=1)
print(f"Input shape: {test_image.shape}")
print(f"Kernel shape: {test_kernel.shape}")
print(f"Conv output shape: {conv_output.shape}")

pooled = max_pool2d(conv_output, pool_size=2, stride=2)
print(f"After max pooling: {pooled.shape}")

## AlexNet 架构（简化）

原始：227x227x3 → 5 个卷积层 → 3 个 FC 层 → 1000 个类

我们针对 32x32 图像的简化版本

In [ ]:
class AlexNetSimplified:
    def __init__(self, num_classes=10):
        """用于 32x32 图像的简化 AlexNet（如 CIFAR-10）
        
        架构：
        - Conv1：3x3x3 -> 32 个滤波器
        - 最大池
        - Conv2：32 -> 64 个滤波器
        - 最大池
        - Conv3：64 -> 128 个滤波器
        - FC层"""
        # 卷积层
        self.conv1_filters = np.random.randn(32, 3, 3, 3) * 0.01
        self.conv1_bias = np.zeros(32)
        
        self.conv2_filters = np.random.randn(64, 32, 3, 3) * 0.01
        self.conv2_bias = np.zeros(64)
        
        self.conv3_filters = np.random.randn(128, 64, 3, 3) * 0.01
        self.conv3_bias = np.zeros(128)
        
        # FC 层（转换后：128 * 4 * 4 = 2048）
        self.fc1_weights = np.random.randn(2048, 512) * 0.01
        self.fc1_bias = np.zeros(512)
        
        self.fc2_weights = np.random.randn(512, num_classes) * 0.01
        self.fc2_bias = np.zeros(num_classes)
    
    def forward(self, x, use_dropout=False, dropout_rate=0.5):
        """执行前向传播。
        x: (3, 32, 32) 图像"""
        # Conv1 + ReLU + MaxPool
        conv1 = conv2d(x, self.conv1_filters, stride=1, padding=1)
        conv1 += self.conv1_bias[:, np.newaxis, np.newaxis]
        conv1 = relu(conv1)
        pool1 = max_pool2d(conv1, pool_size=2, stride=2)  # 32 x 16 x 16
        
        # Conv2 + ReLU + MaxPool
        conv2 = conv2d(pool1, self.conv2_filters, stride=1, padding=1)
        conv2 += self.conv2_bias[:, np.newaxis, np.newaxis]
        conv2 = relu(conv2)
        pool2 = max_pool2d(conv2, pool_size=2, stride=2)  # 64 x 8 x 8
        
        # Conv3 + ReLU + MaxPool
        conv3 = conv2d(pool2, self.conv3_filters, stride=1, padding=1)
        conv3 += self.conv3_bias[:, np.newaxis, np.newaxis]
        conv3 = relu(conv3)
        pool3 = max_pool2d(conv3, pool_size=2, stride=2)  # 128 x 4 x 4
        
        # 展平
        flattened = pool3.reshape(-1)
        
        # FC1 + ReLU + Dropout
        fc1 = np.dot(flattened, self.fc1_weights) + self.fc1_bias
        fc1 = relu(fc1)
        
        if use_dropout:
            dropout_mask = (np.random.rand(*fc1.shape) > dropout_rate).astype(float)
            fc1 = fc1 * dropout_mask / (1 - dropout_rate)
        
        # FC2（输出）
        output = np.dot(fc1, self.fc2_weights) + self.fc2_bias
        
        return output

# 创建模型
alexnet = AlexNetSimplified(num_classes=10)
print("AlexNet (simplified) created")

# 测试前向传递
test_img = np.random.randn(3, 32, 32)
output = alexnet.forward(test_img)
print(f"Input: (3, 32, 32)")
print(f"Output: {output.shape} (class scores)")

## 生成合成图像数据

In [ ]:
def generate_simple_images(num_samples=100, image_size=32):
    """生成具有不同图案的简单合成图像
    类别：
    0：横条纹
    1：竖条纹
    2：斜条纹
    3：棋盘
    4：圆圈
    5：正方形
    6：十字
    7: 三角形
    8：随机噪声
    9：纯色"""
    X = []
    y = []
    
    for i in range(num_samples):
        class_label = i % 10
        img = np.zeros((3, image_size, image_size))
        
        if class_label == 0:  # 横条纹
            for row in range(0, image_size, 4):
                img[:, row:row+2, :] = 1
        
        elif class_label == 1:  # 竖条纹
            for col in range(0, image_size, 4):
                img[:, :, col:col+2] = 1
        
        elif class_label == 2:  # 对角线
            for i in range(image_size):
                if i < image_size:
                    img[:, i, i] = 1
                    if i+1 < image_size:
                        img[:, i, i+1] = 1
        
        elif class_label == 3:  # 棋盘
            for i in range(0, image_size, 4):
                for j in range(0, image_size, 4):
                    if (i//4 + j//4) % 2 == 0:
                        img[:, i:i+4, j:j+4] = 1
        
        elif class_label == 4:  # 圆圈
            center = image_size // 2
            radius = image_size // 3
            y_grid, x_grid = np.ogrid[:image_size, :image_size]
            mask = (x_grid - center)**2 + (y_grid - center)**2 <= radius**2
            img[:, mask] = 1
        
        elif class_label == 5:  # 正方形
            margin = image_size // 4
            img[:, margin:-margin, margin:-margin] = 1
        
        elif class_label == 6:  # 十字形
            mid = image_size // 2
            thickness = 3
            img[:, mid-thickness:mid+thickness, :] = 1
            img[:, :, mid-thickness:mid+thickness] = 1
        
        elif class_label == 7:  # 三角形
            for i in range(image_size):
                width = int((i / image_size) * image_size / 2)
                start = image_size // 2 - width
                end = image_size // 2 + width
                img[:, i, start:end] = 1
        
        elif class_label == 8:  # 随机噪声
            img = np.random.rand(3, image_size, image_size)
        
        else:  # 纯色
            img[:] = 0.7
        
        # 添加颜色变化
        color = np.random.rand(3, 1, 1)
        img = img * color
        
        # 添加噪音
        img += np.random.randn(3, image_size, image_size) * 0.1
        img = np.clip(img, 0, 1)
        
        X.append(img)
        y.append(class_label)
    
    return np.array(X), np.array(y)

# 生成数据集
X_train, y_train = generate_simple_images(200)
X_test, y_test = generate_simple_images(50)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# 可视化样本
class_names = ['H-Stripes', 'V-Stripes', 'Diagonal', 'Checker', 'Circle', 
               'Square', 'Cross', 'Triangle', 'Noise', 'Solid']

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for i in range(10):
    # 查找每个类的第一次出现
    idx = np.where(y_train == i)[0][0]
    img = X_train[idx].transpose(1, 2, 0)  # CHW -> HWC
    axes[i].imshow(img)
    axes[i].set_title(class_names[i])
    axes[i].axis('off')

plt.suptitle('Synthetic Image Dataset (10 Classes)', fontsize=14)
plt.tight_layout()
plt.show()

## 数据增强

AlexNet 广泛使用数据增强——一项关键创新

In [ ]:
def random_flip(img):
    '水平翻转'
    if np.random.rand() > 0.5:
        return img[:, :, ::-1].copy()
    return img

def random_crop(img, crop_size=28):
    '随机裁剪'
    _, h, w = img.shape
    top = np.random.randint(0, h - crop_size + 1)
    left = np.random.randint(0, w - crop_size + 1)
    
    cropped = img[:, top:top+crop_size, left:left+crop_size]
    
    # 调整回原来大小
    # 简单的最近邻（用于演示）
    scale_h = h / crop_size
    scale_w = w / crop_size
    
    resized = np.zeros_like(img)
    for i in range(h):
        for j in range(w):
            src_i = min(int(i / scale_h), crop_size - 1)
            src_j = min(int(j / scale_w), crop_size - 1)
            resized[:, i, j] = cropped[:, src_i, src_j]
    
    return resized

def add_noise(img, noise_level=0.05):
    '添加高斯噪声'
    noise = np.random.randn(*img.shape) * noise_level
    return np.clip(img + noise, 0, 1)

def augment_image(img):
    '应用随机增强'
    img = random_flip(img)
    img = random_crop(img)
    img = add_noise(img)
    return img

# 展示增强效果
original = X_train[0]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

axes[0, 0].imshow(original.transpose(1, 2, 0))
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

for i in range(1, 8):
    augmented = augment_image(original.copy())
    row = i // 4
    col = i % 4
    axes[row, col].imshow(augmented.transpose(1, 2, 0))
    axes[row, col].set_title(f'Augmented {i}')
    axes[row, col].axis('off')

plt.suptitle('Data Augmentation Examples', fontsize=14)
plt.tight_layout()
plt.show()

## 可视化模型学到的滤波器

AlexNet 的见解之一：可视化网络学习的内容

In [ ]:
# 可视化第一层滤波器
filters = alexnet.conv1_filters  # 形状：(32,3,3,3)

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
axes = axes.flatten()

for i in range(min(32, len(axes))):
    # 对滤波器归一化，以便可视化
    filt = filters[i].transpose(1, 2, 0)  # CHW -> HWC
    filt = (filt - filt.min()) / (filt.max() - filt.min() + 1e-8)
    
    axes[i].imshow(filt)
    axes[i].axis('off')
    axes[i].set_title(f'F{i}', fontsize=8)

plt.suptitle('Conv1 Filters (32 filters, 3x3, RGB)', fontsize=14)
plt.tight_layout()
plt.show()

print("These filters learn to detect edges, colors, and simple patterns")

## 特征图可视化

In [ ]:
# 处理图像并可视化特征图
test_image = X_train[4]  # 圆圈

# 通过第一个卷积层进行前向传播
conv1_output = conv2d(test_image, alexnet.conv1_filters, stride=1, padding=1)
conv1_output += alexnet.conv1_bias[:, np.newaxis, np.newaxis]
conv1_output = relu(conv1_output)

# 可视化
fig = plt.figure(figsize=(16, 10))

# 原图
ax = plt.subplot(6, 6, 1)
ax.imshow(test_image.transpose(1, 2, 0))
ax.set_title('Input Image', fontsize=10)
ax.axis('off')

# 特征图
for i in range(min(32, 35)):
    ax = plt.subplot(6, 6, i+2)
    ax.imshow(conv1_output[i], cmap='viridis')
    ax.set_title(f'Map {i}', fontsize=8)
    ax.axis('off')

plt.suptitle('Feature Maps after Conv1 + ReLU', fontsize=14)
plt.tight_layout()
plt.show()

print("Different feature maps respond to different patterns in the image")

## 测试分类

In [ ]:
def softmax(x):
    exp_x = np.exp(x - np.max(x))
    return exp_x / exp_x.sum()

# 在几张图像上进行测试
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for i in range(10):
    idx = i * 5  # 每 5 张图像采样一次
    img = X_test[idx]
    true_label = y_test[idx]
    
    # 前向传播
    logits = alexnet.forward(img, use_dropout=False)
    probs = softmax(logits)
    pred_label = np.argmax(probs)
    
    # 展示
    axes[i].imshow(img.transpose(1, 2, 0))
    axes[i].set_title(f'True: {class_names[true_label]}\nPred: {class_names[pred_label]}\nConf: {probs[pred_label]:.2f}',
                     fontsize=9)
    axes[i].axis('off')

plt.suptitle('AlexNet Predictions (Untrained)', fontsize=14)
plt.tight_layout()
plt.show()

print("Note: Model is untrained, so predictions are random!")
print("Training would require gradient descent, which we've simplified for clarity.")

## 要点

### AlexNet 创新 (2012)：

1. **ReLU 激活**：比 sigmoid/tanh 快得多
   - 正值不饱和
   - 训练速度更快（是 tanh 的 6 倍）

2. **Dropout**：强大的正则化
   - 防止过拟合
   - 用于 FC 层（0.5 速率）

3. **数据增强**：
   - 随机裁剪和翻转
   - 颜色抖动
   - 人为地增加数据集大小

4. **GPU 训练**：
   - 使用 2 个 GTX 580 GPU
   - 支持深度网络的训练

5. **局部响应归一化（LRN）**：
   - 特征图之间的横向抑制
   - 现在不太常见（Batch Norm 取代了它）

### 网络架构
```
Input (227x227x3)
  ↓
Conv1 (96 filters, 11x11, stride 4) + ReLU + MaxPool
  ↓
Conv2 (256 filters, 5x5) + ReLU + MaxPool
  ↓
Conv3 (384 filters, 3x3) + ReLU
  ↓
Conv4 (384 filters, 3x3) + ReLU
  ↓
Conv5 (256 filters, 3x3) + ReLU + MaxPool
  ↓
FC6 (4096) + ReLU + Dropout
  ↓
FC7 (4096) + ReLU + Dropout
  ↓
FC8 (1000 classes) + Softmax
```

### 影响：
- **赢得 ImageNet 2012**：top-5 错误率为 15.3%（第二名为 26.2%）
- **重新激发深度学习研究**：证明深度、数据和计算能力相结合确实有效
- **GPU 革命**：使 GPU 对深度学习至关重要
- **启发现代 CNN**：VGG、ResNet 等架构都建立在这些思想之上

### 为什么它有效：
1. 深层架构（在 2012 年，8 层已经属于深层网络）
2. 大型数据集（120 万 ImageNet 图像）
3. GPU 加速（使训练变得可行）
4. 智能正则化（dropout + 数据增强）
5. ReLU 激活（更快的训练）

### 现代视角：
- AlexNet 现在被认为是“简单”
- ResNet 可以达到 100 层以上
- Batch Norm 取代了 LRN
- 但核心思想依然存在：
  - 深层层次特征
  - 使用卷积建模空间结构
  - 数据增强
  - 正则化